# CPMC with an MPS trial and Slater-determinant walkers

This notebook explains, step by step, what `mps_cpmc_new.py` does. The walkers are Slater determinants and are propagated exactly as in `cpmc.ipynb`. The trial is a DMRG matrix product state (MPS),

\begin{equation*}
\ket{\psi_T} = \sum_{\mathbf n} \Psi_T(\mathbf n)\ket{\mathbf n},\qquad \Psi_T(\mathbf n) = A^{[0]}_{l_0}A^{[1]}_{l_1}\cdots A^{[L-1]}_{l_{L-1}}.
\end{equation*}

Walkers and trial meet in only two places. The overlap $\bra{\psi_T}\phi\rangle$ drives importance sampling and the constraint. The local energy $\bra{\psi_T}H\ket{\phi}/\bra{\psi_T}\phi\rangle$ is what is measured. `notes.pdf` lists ways to evaluate an MPS–determinant overlap. This code uses method 2 there: each walker is turned into an MPS with the Fishman–White construction (PRB **92**, 075132 (2015)), truncated, and contracted with the trial. Per walker:

1. **QR.** $\phi_\sigma = Q_\sigma R_\sigma$ per spin, so $\ket{\phi} = \det R_\uparrow\det R_\downarrow\,\ket{\mathrm{SD}(Q_\uparrow,Q_\downarrow)}$ (§3).
2. **Fishman–White.** Nearest-neighbour Givens rotations $V_\sigma$ turn $Q_\sigma$ into an occupation state: $\ket{\mathrm{SD}(Q_\sigma)} = g_\sigma\,\hat V_\sigma^\dagger\ket{\mathbf n^0_\sigma}$ (§3).
3. **Gates on an MPS.** $\hat V^\dagger_\sigma$ is a circuit of two-site gates. It is applied to $\ket{\mathbf n^0_\sigma}$ as an MPS, truncating the bond dimension to $\chi_w$ (§4–5).
4. **Two spins, one MPS.** The $\uparrow$ and $\downarrow$ channel MPSs are merged into the trial's local basis, with a fermionic reordering sign (§6).
5. **Contraction** with $\psi_T$ for the overlap, and with $H\ket{\psi_T}$ for the local energy (§7–8).
6. **CPMC step.** One conversion per step serves all $2L$ field decisions of the site loop (§9).

Everything is demonstrated on a small chain, $L=8$ at half filling. There the overlap can also be computed by brute force over all $\binom{8}{4}^2 = 4900$ determinant pairs, and every step below is checked against it.

> **Where the approximations are.** Every step above is exact to machine precision, *except*:
>
> | | what | where | exact when | knob |
> |---|---|---|---|---|
> | **A** | Fishman–White block sizes | §3.4 | always with `rank_exact`; with `adaptive` only for the reference determinant | `orbital_plan` |
> | **B** | walker bond truncation | §5.1 | `walker_channel_chi=None` | `walker_channel_chi` |
> | **C** | kept counts per charge block frozen on the reference | §5.3–5.4 | as B | (part of B) |
> | **D** | the site loop reuses the walker MPS of the start of the step | §9 | as B | (consequence of B) |
>
> Two more are not specific to this code: the quality of the trial ($\chi_T$ sets how good the constraint is, not how accurate the overlaps are), and the Trotter and constraint errors of CPMC itself. §10 lists the measured size of each.

In [1]:
import contextlib
import os
import sys
from itertools import combinations

import numpy as np
import scipy.linalg
import jax
import jax.numpy as jnp

import mps_cpmc_new as m

np.set_printoptions(precision=4, suppress=True, linewidth=110)
rng = np.random.default_rng(0)


@contextlib.contextmanager
def quiet():
    # silence output written directly to file descriptor 1 (pyblock3 prints from compiled code)
    sys.stdout.flush()
    saved = os.dup(1)
    with open(os.devnull, "w") as devnull:
        os.dup2(devnull.fileno(), 1)
        try:
            yield
        finally:
            os.dup2(saved, 1)
            os.close(saved)

L, N = 8, 4                          # sites, electrons per spin (half filling)
t, U, dt = 1.0, 4.0, 0.01
h1 = m.hopping_matrix(L, t)          # open chain
C = np.linalg.eigh(h1)[1][:, :N]     # RHF orbitals: the reference determinant, for both spins

## 1. The trial, and the conventions everything relies on

The trial comes from pyblock3 (`run_dmrg`). `densify_with_charges` turns it into dense tensors $A^{[x]}_{a,l,b}$ with local index $l = n_{x\uparrow} + 2n_{x\downarrow}$:

| $l$ | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| state | $\ket{0}$ | $c^\dagger_{x\uparrow}\ket{0}$ | $c^\dagger_{x\downarrow}\ket{0}$ | $c^\dagger_{x\uparrow}c^\dagger_{x\downarrow}\ket{0}$ |

Fermion operators are ordered site by site, $\uparrow$ before $\downarrow$ on each site, with $x$ increasing from left to right:

\begin{equation*}
\ket{\mathbf n} = \prod_{x=0}^{L-1}\big(c^\dagger_{x\uparrow}\big)^{n_{x\uparrow}}\big(c^\dagger_{x\downarrow}\big)^{n_{x\downarrow}}\ket{0}.
\end{equation*}

Every bond index of the trial also carries a **charge** $(N_\uparrow, N_\downarrow)$, the number of electrons of each spin on the sites to its left. The MPS conserves particle number, so $A^{[x]}_{a,l,b}$ can only be nonzero if the charge of $b$ is the charge of $a$ plus the $(n_\uparrow, n_\downarrow)$ of $l$. §4 and §6 use these labels.

The check below is the one that matters for the conventions. `hubbard_mpo` builds $H$ in the convention above, and its expectation value in the densified MPS must reproduce pyblock3's DMRG energy. A wrong operator order, or a wrong sign on $c^\dagger_\uparrow c^\dagger_\downarrow\ket 0$, would change hopping matrix elements and show up at the scale of $t$.

In [2]:
cfg = m.Config(L=L, n_up=N, n_down=N, hopping=t, interaction=U, trial_chi=16, dmrg_sweeps=10)
with quiet():
    dmrg_mps, E_dmrg = m.run_dmrg(m.build_dmrg_hamiltonian(cfg), cfg)
trial, trial_charges = m.densify_with_charges(dmrg_mps, L)

print("trial bond dimensions:", [A.shape[0] for A in trial] + [trial[-1].shape[2]])
print("charges (N_up, N_down) present on bond 4:", sorted(set(map(tuple, trial_charges[4].tolist()))))


def mps_dot(A, B):
    # <A|B> for two real MPS, each a list of (D_left, d, D_right) arrays
    env = np.ones((1, 1))
    for a, b in zip(A, B):
        env = np.einsum("ab,asc,bsd->cd", env, np.asarray(a), np.asarray(b))
    return env[0, 0]


H_trial = m.compress_mps(m.apply_mpo(m.hubbard_mpo(L, t, U), trial))    # H|psi_T>, as an MPS
E_trial = mps_dot(trial, H_trial) / mps_dot(trial, trial)
print(f"pyblock3 DMRG energy                  {E_dmrg:.10f}")
print(f"<T|H|T>/<T|T> with our MPO/convention {E_trial:.10f}")

trial bond dimensions: [1, 4, 16, 16, 16, 16, 16, 4, 1]
charges (N_up, N_down) present on bond 4: [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3), (3, 1), (3, 2), (3, 3)]
pyblock3 DMRG energy                  -4.2345446046
<T|H|T>/<T|T> with our MPO/convention -4.2334907824


The two differ by about $10^{-3}$, because pyblock3 reports the energy of its last two-site wavefunction before that is truncated to $\chi_T=16$. That is three orders of magnitude below the scale of $t$, where a convention error would show, so the conventions of pyblock3 and of this code are the same. From here on the trial is the densified MPS, and its energy is the second number.

## 2. The overlap by brute force

The walker is an unrestricted determinant with all $\uparrow$ electrons created first,

\begin{equation*}
\ket{\phi} = \prod_{i=1}^{N_\uparrow}\Big(\sum_x \phi_{\uparrow,xi}\,c^\dagger_{x\uparrow}\Big)\prod_{j=1}^{N_\downarrow}\Big(\sum_y \phi_{\downarrow,yj}\,c^\dagger_{y\downarrow}\Big)\ket{0}.
\end{equation*}

Its amplitude on the occupation state with $\uparrow$ electrons on sites $S_\uparrow$ and $\downarrow$ electrons on $S_\downarrow$ is a product of two minors (`msd_fast_update.ipynb`) times a sign. Reordering the creation operators from "all $\uparrow$, then all $\downarrow$" to the site order of §1 moves each $\uparrow$ electron past every $\downarrow$ electron on a site to its left:

\begin{equation*}
\phi(\mathbf n) = (-1)^{K(\mathbf n)}\det\phi_\uparrow[S_\uparrow,:]\,\det\phi_\downarrow[S_\downarrow,:],\qquad K(\mathbf n) = \#\{(x,y):\ x\in S_\uparrow,\ y\in S_\downarrow,\ y<x\}.
\end{equation*}

So, with everything real,

\begin{equation*}
\bra{\psi_T}\phi\rangle = \sum_{S_\uparrow,S_\downarrow}\Psi_T(\mathbf n)\,(-1)^{K(\mathbf n)}\det\phi_\uparrow[S_\uparrow,:]\,\det\phi_\downarrow[S_\downarrow,:].
\end{equation*}

The sum has $\binom{L}{N_\uparrow}\binom{L}{N_\downarrow}$ terms, which is why the rest of this notebook exists. At $L=8$ it is 4900 terms, and it is the reference for every check. The same sum with $\Psi_T$ replaced by the amplitudes of $H\ket{\psi_T}$ gives $\bra{\psi_T}H\ket{\phi}$.

The walkers used throughout are the RHF determinant propagated for 100 steps by the CPMC propagator with random, unguided fields. They have drifted well away from HF, and they are deliberately left non-orthonormal.

In [3]:
strings = np.array(list(combinations(range(L), N)))       # occupied sites of one spin, sorted
occ = np.zeros((len(strings), L), int)
occ[np.arange(len(strings))[:, None], strings] = 1        # occ[k, x] = n_x of string k


def amplitude(tensors, n_up, n_down):
    # <n|MPS> in the local basis l = n_up + 2 n_down
    v = np.ones((1, 1))
    for A, a, b in zip(tensors, n_up, n_down):
        v = v @ np.asarray(A)[:, a + 2 * b, :]
    return v[0, 0]


def reorder_sign(n_up, n_down):
    # (-1)^K, K = number of (up at x, down at y) pairs with y < x
    return (-1) ** sum(n_up[x] * n_down[:x].sum() for x in range(L))


def signed_amplitudes(tensors):
    return np.array([[reorder_sign(a, b) * amplitude(tensors, a, b) for b in occ] for a in occ])


Psi_T, HPsi_T = signed_amplitudes(trial), signed_amplitudes(H_trial)


def exact_overlap(ca, cb):
    # <psi_T|phi> by brute force
    return np.linalg.det(ca[strings]) @ Psi_T @ np.linalg.det(cb[strings])


def exact_energy(ca, cb):
    da, db = np.linalg.det(ca[strings]), np.linalg.det(cb[strings])
    return (da @ HPsi_T @ db) / (da @ Psi_T @ db)


def random_walker(L, N, steps, seed):
    # HF propagated by the CPMC propagator with random (unguided) HS fields; not orthonormalized
    rng = np.random.default_rng(seed)
    h = m.hopping_matrix(L, t)
    half = scipy.linalg.expm(-0.5 * dt * h)
    gamma = np.arccosh(np.exp(0.5 * dt * U))
    ca = cb = np.linalg.eigh(h)[1][:, :N]
    for _ in range(steps):
        s = rng.choice([-1.0, 1.0], L)
        ca = half @ (np.exp(gamma * s)[:, None] * (half @ ca))
        cb = half @ (np.exp(-gamma * s)[:, None] * (half @ cb))
    return ca, cb


walkers = [random_walker(L, N, steps=100, seed=k) for k in range(6)]
print(f"{len(strings)}^2 = {len(strings) ** 2} determinant pairs")
print("exact overlaps of the six walkers:", np.array([exact_overlap(*w) for w in walkers]))

70^2 = 4900 determinant pairs
exact overlaps of the six walkers: [-9.0490e+06 -1.0509e+06 -9.5491e+06 -3.3774e+07 -2.8168e+09 -8.2801e+07]


## 3. From a determinant to a circuit: Fishman–White

### 3.1 The correlation matrix

Take one spin channel and orthonormalize it first, $\phi = QR$. Then $\ket{\mathrm{SD}(\phi)} = \det R\,\ket{\mathrm{SD}(Q)}$, because the triangular $R$ only mixes the orbitals among themselves. $\det R$ is kept aside and multiplies the overlap at the end (§7).

A determinant is fixed by its correlation matrix

\begin{equation*}
\Lambda_{xy} = \bra{\mathrm{SD}}c^\dagger_x c_y\ket{\mathrm{SD}} = (QQ^T)_{xy}.
\end{equation*}

$\Lambda$ is a projector: its eigenvalues are exactly 0 (empty modes) and 1 (occupied modes). Restricted to a block of a few sites it is not a projector any more. Its eigenvalues lie between 0 and 1, and a mode with an eigenvalue strictly inside is entangled with the rest of the chain. But **an eigenvalue exactly 0 or 1 of a block is a mode that lives on those sites alone and is empty or filled.** It factorizes out of the determinant.

For the first $B$ sites, $\Lambda_B = Q_BQ_B^T$, where $Q_B$ is made of the top $B$ rows of $Q$. Its rank is at most $N$, so for $B\ge N+1$ it has an exact zero eigenvalue. The same argument on $1-\Lambda$ gives an exact eigenvalue 1 once $B\ge L-N+1$. This rank count is what makes the `rank_exact` plan exact for every walker (§3.4).

In [4]:
ca, cb = walkers[0]
Q, R = np.linalg.qr(ca)                     # |SD(ca)> = det(R) |SD(Q)>
Lam = Q @ Q.T
print("eigenvalues of the whole Lambda :", np.linalg.eigvalsh(Lam))
print("eigenvalues of the 3-site block :", np.linalg.eigvalsh(Lam[:3, :3]))
print(f"eigenvalues of the {N + 1}-site block :", np.linalg.eigvalsh(Lam[:N + 1, :N + 1]))

eigenvalues of the whole Lambda : [-0. -0.  0.  0.  1.  1.  1.  1.]
eigenvalues of the 3-site block : [0.0018 0.2781 0.9999]
eigenvalues of the 5-site block : [-0.      0.0124  0.7086  0.9998  1.    ]


### 3.2 One step: move a decoupled mode onto one site

Let $v$ be the eigenvector of $\Lambda_B$ with eigenvalue 0 (or 1). A sequence of $B-1$ nearest-neighbour Givens rotations, from the bottom of the block up, collects all of $v$ on the first site of the block. The rotation of rows $(p, p+1)$ by $\tan\theta = v_{p+1}/v_p$ zeroes $v_{p+1}$. Applying the same rotations to the rows of $Q$ turns row $k$ into the mode itself. So row $k$ of the rotated $Q$ is zero for an empty mode (and has norm one for a filled mode), and site $k$ no longer talks to the others.

The function below is `_rotate_mode_to_front` of the module, written out.

In [5]:
def rotate_to_front(rows, start, v):
    # adjacent Givens rotations that move the mode v (on rows start..start+B-1) onto row `start`;
    # rotates `rows` in place and returns the gates as (site p, angle theta)
    v, gates = v.copy(), []
    for j in range(len(v) - 1, 0, -1):
        theta = np.arctan2(v[j], v[j - 1])
        c, s = np.cos(theta), np.sin(theta)
        v[j - 1], v[j] = c * v[j - 1] + s * v[j], 0.0
        p = start + j - 1
        rows[[p, p + 1]] = np.array([[c, s], [-s, c]]) @ rows[[p, p + 1]]
        gates.append((p, theta))
    return gates


B = N + 1
values, vectors = np.linalg.eigh(Lam[:B, :B])
rows = Q.copy()
gates = rotate_to_front(rows, 0, vectors[:, 0])          # eigenvalue 0: an empty mode
print("gates (p, theta):", [(p, round(th, 3)) for p, th in gates])
print("row 0 after the rotations:", rows[0])
print("site 0 decoupled from the rest:", np.allclose((rows @ rows.T)[0], 0))

gates (p, theta): [(3, np.float64(3.059)), (2, np.float64(0.221)), (1, np.float64(1.693)), (0, np.float64(1.461))]
row 0 after the rotations: [-0.  0.  0.  0.]
site 0 decoupled from the rest: True


### 3.3 The whole plan, and the gauge

Repeating this for $k = 0, 1, \dots, L-2$ on the remaining sites gives a product of rotations $V$ such that every row of $VQ$ is either zero or has norm one. The rotated determinant is then a single occupation state,

\begin{equation*}
\ket{\mathrm{SD}(VQ)} = g\,\ket{\mathbf n^0},\qquad g = \det\big((VQ)[\text{occupied rows},:]\big) = \pm1 .
\end{equation*}

With $\hat V$ the many-body operator of the orbital rotation, $\hat V\ket{\mathrm{SD}(Q)} = \ket{\mathrm{SD}(VQ)}$, so

\begin{equation*}
\ket{\mathrm{SD}(Q)} = g\,\hat V^\dagger\ket{\mathbf n^0}.
\end{equation*}

The code splits this into two parts.

* `make_orbital_plan(C_ref, mode)` runs once, on the reference determinant, and fixes the **structure**: the block size $B$ of every step, whether the isolated mode is empty or filled, and a reference eigenvector used only to keep the eigenvector's sign continuous.
* `channel_angles(Q, plan)` recomputes the **numerical angles** for one walker, with that structure frozen. A CPMC run never calls it separately: `channel_mps` calls it at the start of every conversion (with `jnp`, inside the jitted step), so the angles always belong to the current walker. A frozen structure means static shapes, which is what lets JAX compile the conversion once.

The cell below calls `channel_angles` directly, with NumPy, only to look inside one conversion. It prints the angles and the rotated rows $VQ$ for walker 0, and §3.5 reuses those angles for the dense check. Nothing from this call is stored. The only NumPy use in the module is in `plan_bonds`, which replays the angles of the reference determinant for the truncation dry run (§5.3).

| computed once, on the reference (NumPy) | computed for every walker, every conversion (JAX) |
|---|---|
| `make_orbital_plan`: block sizes, isolated modes, reference vectors | `qr_with_det`: $Q$ and $\det R$ |
| `plan_bonds`: kept counts per charge block, bond labels | `channel_angles`: this walker's angles and $VQ$ |
| contraction layout with the trial | gates, centre moves, truncated splits (`channel_mps`) |
| | gauge $g$, contraction with the trial |

In [6]:
plan = m.make_orbital_plan(C, "rank_exact")            # structure, from the reference
angles, rotated = m.channel_angles(Q, plan, xp=np)      # display only: channel_mps recomputes these per walker
rotated = np.stack(rotated)
g = np.linalg.det(rotated[plan.occupation == 1])

print("block size B per step:", plan.block_sizes)
print("occupations n0       :", plan.occupation)
print("number of gates      :", len(angles))
print("norms of the rows of VQ:", np.linalg.norm(rotated, axis=1))
print("gauge g =", g)

block size B per step: [5 5 5 5 1 1 1]
occupations n0       : [0 0 0 0 1 1 1 1]
number of gates      : 16
norms of the rows of VQ: [0. 0. 0. 0. 1. 1. 1. 1.]
gauge g = -1.0


### 3.4 ⚠ Approximation A: the block sizes

The two plans differ in the block size of each step.

* **`rank_exact`** uses $B = \min(N, L-N)+1$. By the rank argument of §3.1, every block has an exact 0 (or 1) eigenvalue **for any walker**, so $\ket{\mathrm{SD}(VQ)} = g\ket{\mathbf n^0}$ holds exactly.
* **`adaptive`** grows $B$ only until the *reference* determinant has an eigenvalue within `occupation_tolerance` ($10^{-10}$) of 0 or 1. For a walker, the same small block has no exactly pure mode, so $VQ$ is not exactly an occupation state. The conversion then keeps only the $\ket{\mathbf n^0}$ component of $\hat V\ket{\mathrm{SD}(Q)}$ and drops a norm $1-g^2$, now with $|g|<1$.

For small $L$ the plans coincide, because the rank bound is reached before any mode of the reference becomes pure. At larger $L$, `adaptive` needs far fewer gates. The norm it drops stays small because walkers keep the short-range correlation structure of the reference. The cell measures both at $L=8$ and at $L=24$ (plans only, no MPS needed).

In [7]:
def rank_bound(plan):
    # min(N_left, holes_left) + 1 at each step: the block size that guarantees an exact mode
    n_left = plan.occupation.sum()
    h_left, out = len(plan.occupation) - n_left, []
    for o in plan.occupation[:-1]:
        out.append(min(n_left, h_left) + 1)
        n_left, h_left = n_left - o, h_left - (1 - o)
    return np.array(out)


def dropped_norm(C_ref, walker, mode):
    # 1 - g^2: the norm the Gaussian step throws away for this walker
    plan = m.make_orbital_plan(C_ref, mode)
    _, rows = m.channel_angles(np.linalg.qr(walker)[0], plan, xp=np)
    return 1 - np.linalg.det(np.stack(rows)[plan.occupation == 1]) ** 2


for L_big in (8, 24):
    C_big = np.linalg.eigh(m.hopping_matrix(L_big, t))[1][:, :L_big // 2]
    walker_big = random_walker(L_big, L_big // 2, steps=100, seed=1)[0]
    adaptive = m.make_orbital_plan(C_big, "adaptive")
    print(f"L = {L_big}:  adaptive B {adaptive.block_sizes}")
    print(f"         rank bound {rank_bound(adaptive)}")
    for mode in ("rank_exact", "adaptive"):
        gates = int((m.make_orbital_plan(C_big, mode).block_sizes - 1).sum())
        print(f"         {mode:10s}: {gates:4d} gates, dropped norm on a drifted walker "
              f"{dropped_norm(C_big, walker_big, mode):.1e}")

L = 8:  adaptive B [5 4 4 3 3 2 2]
         rank bound [5 4 4 3 3 2 2]
         rank_exact:   16 gates, dropped norm on a drifted walker 0.0e+00
         adaptive  :   16 gates, dropped norm on a drifted walker 2.2e-16
L = 24:  adaptive B [8 7 9 8 9 8 9 8 9 8 8 7 7 6 6 5 5 4 4 3 3 2 2]
         rank bound [13 12 12 11 11 10 10  9  9  8  8  7  7  6  6  5  5  4  4  3  3  2  2]
         rank_exact:  144 gates, dropped norm on a drifted walker 2.2e-16
         adaptive  :  122 gates, dropped norm on a drifted walker 1.4e-10


### 3.5 The many-body gates

A rotation $G = \begin{pmatrix}c & s\\ -s & c\end{pmatrix}$ of rows $p, p+1$ ($Q\to GQ$) acts on creation operators as $c^\dagger_q\to\sum_{p'}G_{p'q}c^\dagger_{p'}$. On the two-site states $\ket{n_pn_{p+1}}$:

\begin{equation*}
\ket{00}\to\ket{00},\qquad \ket{10}\to c\ket{10}-s\ket{01},\qquad \ket{01}\to s\ket{10}+c\ket{01},\qquad \ket{11}\to \det G\,\ket{11}=\ket{11}.
\end{equation*}

There is no Jordan–Wigner string, because the two sites are neighbours in the fermion order. We need $\hat V^\dagger$: each gate's inverse (the transposed rotation), in reverse order. Since $\hat V = \hat G_M\cdots\hat G_1$ (the first planned gate acts first), $\hat V^\dagger\ket{\mathbf n^0} = \hat G_1^\dagger\cdots\hat G_M^\dagger\ket{\mathbf n^0}$, and the last planned gate acts first.

The cell does this on a dense state vector of all $2^L$ occupations of one spin channel, the most direct check possible: $g\,\hat V^\dagger\ket{\mathbf n^0}$ must reproduce every minor $\det Q[S,:]$.

In [8]:
def two_site_gate(theta):
    # G^dagger on |n_p n_p+1>, basis order 00, 01, 10, 11 (index 2 n_p + n_p+1)
    c, s = np.cos(theta), np.sin(theta)
    gate = np.eye(4)
    gate[1:3, 1:3] = [[c, s], [-s, c]]
    return gate


def apply_gate(psi, gate, p):
    # apply a two-site gate to sites p, p+1 of a dense state psi[n_0, ..., n_{L-1}]
    psi = np.moveaxis(psi, (p, p + 1), (0, 1))
    shape = psi.shape
    psi = (gate @ psi.reshape(4, -1)).reshape(shape)
    return np.moveaxis(psi, (0, 1), (p, p + 1))


psi = np.zeros((2,) * L)
psi[tuple(plan.occupation)] = 1.0                      # the occupation state |n0>
for p, theta in reversed(angles):                      # last planned gate first
    psi = apply_gate(psi, two_site_gate(theta), p)

minors = np.array([np.linalg.det(Q[s]) for s in strings])
dense_amps = np.array([psi[tuple(o)] for o in occ])
print(f"g <n|V^dagger|n0> == det Q[S,:] for all {len(strings)} strings:", np.allclose(g * dense_amps, minors))
print("norm outside the N-particle sector:", abs(np.sum(psi ** 2) - np.sum(dense_amps ** 2)))

g <n|V^dagger|n0> == det Q[S,:] for all 70 strings: True
norm outside the N-particle sector: 0.0


## 4. The same circuit on an MPS

`channel_mps` does what the dense vector did, but keeps the state as an MPS with physical dimension 2. A gate on sites $(p,p+1)$ is applied in two steps:

1. contract the two tensors into $\Theta_{a,n_p,n_{p+1},b}$ and apply the $4\times4$ gate of §3.5 (`gate_pair`);
2. reshape $\Theta$ to a matrix $(a,n_p)\times(n_{p+1},b)$ and split it again, $\Theta = A\,B$ (`split_pair`).

The state has a fixed particle number, so every bond carries a charge, the number of electrons to its left, starting from the labels of the occupation state. $\Theta$ is block diagonal in the charge of the middle bond: row $(a,n_p)$ has charge $q_a+n_p$, and column $(n_{p+1},b)$ has charge $q_b - n_{p+1}$. The split is done block by block (`sector_plan`), so every tensor stays charge-labelled.

Without truncation each block keeps its full rank and the MPS is exact. Its bond dimension then grows up to $\sum_q\min\big(\binom{x}{q},\binom{L-x}{N-q}\big)$, which is $2^{L/2}$ in the middle of the chain: harmless at $L=8$, impossible at $L=32$. Hence truncation (§5).

In [9]:
def channel_amplitude(tensors, n):
    # <n|MPS> for a one-spin (d = 2) MPS
    v = np.ones((1, 1))
    for A, x in zip(tensors, n):
        v = v @ np.asarray(A)[:, x, :]
    return v[0, 0]


tensors, charges, g_mps = m.channel_mps(jnp.asarray(Q), plan)            # no truncation
mps_amps = np.array([channel_amplitude(tensors, o) for o in occ])

print("MPS amplitudes equal the dense state:", np.allclose(mps_amps, dense_amps))
print("same gauge g:", np.isclose(float(g_mps), g))
print("bond dimensions:", [A.shape[0] for A in tensors] + [1])
print("charges on bond 4:", charges[4])
p = 3
print(f"charge blocks of a split of sites {p},{p + 1}, as (rows, columns, rank):",
      [(len(r), len(c), k) for r, c, k in m.sector_plan(charges[p], charges[p + 2]).sectors])

MPS amplitudes equal the dense state: True
same gauge g: True
bond dimensions: [1, 2, 4, 8, 16, 8, 4, 2, 1]
charges on bond 4: [0 1 1 1 1 2 2 2 2 2 2 3 3 3 3 4]
charge blocks of a split of sites 3,4, as (rows, columns, rank): [(1, 1, 1), (4, 4, 4), (6, 6, 6), (4, 4, 4), (1, 1, 1)]


## 5. ⚠ Approximation B: truncating the walker MPS

### 5.1 Truncate at Schmidt values: the orthogonality centre must be on the gate

With `walker_channel_chi` $=\chi_w$, each split keeps only $\chi_w$ singular vectors in total, summed over charge blocks, and drops the rest. **This is the approximation of the method.** The walker MPS $\tilde\phi$ is no longer $\phi$, and every overlap and local energy is computed with $\tilde\phi$.

Dropping the smallest singular values of $\Theta$ is the best truncation of that bond only if they are the Schmidt values of the whole state. That requires the rest of the MPS to be an isometry on both sides: every tensor left of the pair left-orthonormal and every tensor right of it right-orthonormal (mixed canonical form, orthogonality centre on the pair). Otherwise the environments distort the singular values, and the discarded weight is not the error.

Within one Fishman–White step the gates run left to right, each split leaves a left-orthonormal tensor behind, and the centre follows along. Between steps it does not: step $k$ starts at site $k$, while the centre was left at the end of the previous block. `channel_mps`, and the dry run `plan_bonds`, therefore **move the orthogonality centre onto every gate before splitting it** (`_move_centre`, with a charge-blocked QR when moving right and an LQ when moving left). Those QRs are exact; only the splits truncate.

The cell compares, at $\chi_w = 4$: the old behaviour (centre never moved), the current code, and the best achievable at the same bond dimensions (the exact MPS compressed once, canonically). It also checks that the converted MPS ends in mixed canonical form.

In [10]:
chi = 4
bond_plan = m.plan_bonds(C, plan, chi)                     # dry run on the reference (§5.3)


def fidelity(exact, approx):
    return mps_dot(exact, approx) ** 2 / (mps_dot(exact, exact) * mps_dot(approx, approx))


@contextlib.contextmanager
def without_centre_moves():
    # the old behaviour: split each gate wherever the orthogonality centre happens to be
    saved = m._move_centre
    m._move_centre = lambda tensors, charges, centre, target, xp=None: target
    try:
        yield
    finally:
        m._move_centre = saved


def canonical_compress(tensors, dims):
    # the best at these bond dimensions: left-canonical QR sweep, then right-to-left SVD truncation
    ts = [np.asarray(A).copy() for A in tensors]
    for i in range(len(ts) - 1):
        Dl, d, Dr = ts[i].shape
        q, r = np.linalg.qr(ts[i].reshape(Dl * d, Dr))
        ts[i], ts[i + 1] = q.reshape(Dl, d, -1), np.tensordot(r, ts[i + 1], (1, 0))
    for i in range(len(ts) - 1, 0, -1):
        Dl, d, Dr = ts[i].shape
        u, s, vt = np.linalg.svd(ts[i].reshape(Dl, d * Dr), full_matrices=False)
        k = min(dims[i], len(s))
        ts[i], ts[i - 1] = vt[:k].reshape(k, d, Dr), np.tensordot(ts[i - 1], u[:, :k] * s[:k], (2, 0))
    return ts


print(f"chi_w = {chi}: 1 - fidelity with the exact channel MPS")
print(f"{'walker':>6} {'centre not moved (old)':>24} {'centre on each gate (now)':>27} {'best possible':>15}")
for k, (wa, _) in enumerate(walkers):
    Qk = jnp.asarray(np.linalg.qr(wa)[0])
    exact_k, _, _ = m.channel_mps(Qk, plan)
    now, now_charges, _ = m.channel_mps(Qk, plan, bond_plan)
    with without_centre_moves():
        old, _, _ = m.channel_mps(Qk, plan, m.plan_bonds(C, plan, chi))
    best = canonical_compress(exact_k, [len(q) for q in now_charges])
    print(f"{k:>6} {1 - fidelity(exact_k, old):24.2e} {1 - fidelity(exact_k, now):27.2e} "
          f"{1 - fidelity(exact_k, best):15.2e}")

centre = angles[0][0] + 1                                  # the last gate applied is the first one planned
defect = 0.0
for i, A in enumerate(map(np.asarray, now)):
    if i < centre:
        M = A.reshape(-1, A.shape[2]); defect = max(defect, np.abs(M.T @ M - np.eye(M.shape[1])).max())
    elif i > centre:
        M = A.reshape(A.shape[0], -1); defect = max(defect, np.abs(M @ M.T - np.eye(M.shape[0])).max())
print(f"\nmixed canonical around site {centre} after conversion: largest isometry defect {defect:.1e}")

chi_w = 4: 1 - fidelity with the exact channel MPS
walker   centre not moved (old)   centre on each gate (now)   best possible
     0                 3.47e-03                    5.19e-04        5.19e-04
     1                 6.09e-02                    1.07e-02        1.28e-03
     2                 5.05e-02                    1.97e-03        5.47e-04
     3                 4.39e-03                    2.65e-04        9.86e-05
     4                 1.15e-04                    3.81e-05        3.81e-05
     5                 2.24e-03                    2.19e-04        1.41e-04

mixed canonical around site 4 after conversion: largest isometry defect 4.4e-16


Moving the centre makes the truncation 3 to 26 times more accurate already at $L=8$, and the gap grows with system size (table at the end of §5). The current code reaches the best achievable value on walkers 0 and 4 and stays within a small factor on the others. §5.4 shows that this remaining gap is Approximation C.

### 5.2 The truncated split: one small eigenproblem per block

`split_pair` does not SVD $\Theta$. A truncated charge block $M$ needs only its top-$k$ left singular subspace, and `_factor_block` gets it from one eigendecomposition of the block's smaller Gram matrix:

\begin{equation*}
MM^T = US^2U^T:\quad M_k = U_k\,\big(U_k^TM\big),\qquad\qquad M^TM = VS^2V^T:\quad M_k = \big(MV_kS_k^{-1}\big)\big(S_kV_k^T\big),
\end{equation*}

the first when $M$ has at most as many rows as columns, the second otherwise. Each gives a left isometry times the new orthogonality centre, and both are the SVD truncation. arXiv:2212.09782 (Unfried, Hauschild, Pollmann) reaches the same subspace with a QR, $M = QR$, followed by an eigendecomposition of $RR^T$. Using the Gram matrix directly gives the same result with one LAPACK call instead of two. Two more cases skip LAPACK entirely or partly, decided from the static plan: a block with a single row or column is factored in closed form, and a block that keeps its full rank uses a plain QR.

Together these made the conversion 22% faster, with 40% less compile time, at $\chi_w=4$ for both $L=32$ and $L=48$, with overlaps unchanged to $\le10^{-11}$. Forming a Gram matrix squares the singular values, as $RR^T$ did, so values below $\sim10^{-8}$ of the largest are not resolved individually. The $M^TM$ branch also divides by the kept singular values. Neither has mattered in practice.

In [11]:
def rank_k(M, k):
    u, s, vt = np.linalg.svd(M, full_matrices=False)
    return (u[:, :k] * s[:k]) @ vt[:k]                  # best rank-k approximation


k = 3
for rows, cols in ((6, 10), (10, 6)):                   # both Gram branches of _factor_block
    M = rng.standard_normal((rows, 6)) @ np.diag(np.logspace(0, -6, 6)) @ rng.standard_normal((6, cols))
    A, B = m._factor_block(M, k, truncate=True, xp=np)
    q, r = np.linalg.qr(M)                              # the QR + eigh(R R^T) route, for comparison
    _, V = np.linalg.eigh(r @ r.T)
    Vk = V[:, ::-1][:, :k]
    print(f"{rows}x{cols} block: Gram truncation equals the SVD one: {np.allclose(np.asarray(A) @ np.asarray(B), rank_k(M, k))}; "
          f"equals QR + eigh(R R^T): {np.allclose(np.asarray(A) @ np.asarray(B), (q @ Vk) @ (Vk.T @ r))}; "
          f"A is an isometry: {np.allclose(np.asarray(A).T @ np.asarray(A), np.eye(k))}")

6x10 block: Gram truncation equals the SVD one: True; equals QR + eigh(R R^T): True; A is an isometry: True
10x6 block: Gram truncation equals the SVD one: True; equals QR + eigh(R R^T): True; A is an isometry: True


### 5.3 How `plan_bonds` keeps the charges and shapes fixed

Every array shape in a conversion follows from the bond labels. §4 showed that each split's matrix is block diagonal in the charge of the middle bond. The new middle bond is the concatenation of the vectors kept in each block, so if block $s$, with charge $q_s$, keeps $k_s$ vectors,

\begin{equation*}
\dim(\text{new bond}) = \sum_s k_s,\qquad \text{labels} = (\underbrace{q_1,\dots,q_1}_{k_1},\ \underbrace{q_2,\dots,q_2}_{k_2},\ \dots).
\end{equation*}

Those labels are the input of the next split and of the centre moves. So the whole chain of shapes is fixed by the list of counts $(k_s)$ at every split, and by nothing else.

* **Without truncation**, $k_s = \min(\text{rows}_s, \text{cols}_s)$, which depends only on the labels. The shapes are the same for every walker automatically. The centre moves (QR and LQ) are always of this kind.
* **With truncation**, "keep the $\chi_w$ largest singular values over all blocks" depends on the walker's singular values. Two walkers could keep, say, $(1,2,1)$ and $(2,1,1)$ in the same split: the same total $\chi_w$, but different labels, and different shapes from then on.

`plan_bonds` removes that dependence. It runs the conversion once, in NumPy, on the reference determinant, with the same gates and the same centre moves. At each split it records the per-block counts the reference would choose (`BondPlan.kept_per_sector`, one tuple per gate), together with the resulting bond labels (`BondPlan.charges`). In a CPMC run, `channel_mps(Q, plan, bond_plan)` looks up the tuple of gate $i$, and `split_pair` keeps exactly $k_s$ vectors in block $s$: the $k_s$ largest *within that block*, recomputed from the walker's own numbers. Every walker then produces tensors with identical shapes and identical labels. That is what lets one `jax.jit`/`jax.vmap` program convert all walkers, and what lets the contraction layout with the trial (`make_contraction_plan`) be built once.

The first cell opens the dry run at the last split of the conversion and prints its charge blocks, their full rank, and how many vectors the reference keeps in each.

In [12]:
print(f"{len(bond_plan.kept_per_sector)} splits, one per gate; bond dimensions after conversion:",
      [len(q) for q in bond_plan.charges])

# the last gate applied is the first one planned; its split is the last operation of the
# conversion, so the final labels on its two outer bonds are the ones it saw
p = angles[0][0]
ql, qr = bond_plan.charges[p], bond_plan.charges[p + 2]
middle = sorted(set((ql[:, None] + np.arange(2)).ravel().tolist())
                & set((qr[None, :] - np.arange(2)[:, None]).ravel().tolist()))
blocks = m.sector_plan(ql, qr).sectors
print(f"\nlast split, sites {p} and {p + 1}:")
print(f"{'middle charge':>13} {'block rows x cols':>18} {'full rank':>10} {'kept':>5}")
for q, (rows_s, cols_s, rank), k in zip(middle, blocks, bond_plan.kept_per_sector[-1]):
    print(f"{q:>13} {f'{len(rows_s)} x {len(cols_s)}':>18} {rank:>10} {k:>5}")
print("labels of the new bond:", bond_plan.charges[p + 1])

16 splits, one per gate; bond dimensions after conversion: [1, 2, 4, 4, 4, 4, 4, 2, 1]

last split, sites 3 and 4:
middle charge  block rows x cols  full rank  kept
            1              1 x 1          1     1
            2              3 x 3          3     2
            3              3 x 3          3     1
            4              1 x 1          1     0
labels of the new bond: [1 2 2 3]


With the counts frozen, walkers that look nothing alike come out with the same shapes and the same labels. So a single vectorized call converts all six at once. It returns one array per site with a leading walker axis, which is possible only because nothing about the shapes depends on the walker.

In [13]:
shapes, same_labels = set(), True
for wa, _ in walkers:
    ts, qn, _ = m.channel_mps(jnp.asarray(np.linalg.qr(wa)[0]), plan, bond_plan)
    shapes.add(tuple(A.shape for A in ts))
    same_labels &= all(np.array_equal(a, b) for a, b in zip(qn, bond_plan.charges))
print("distinct tensor shapes over the six walkers:", len(shapes))
print("every walker's bond labels equal the dry run's:", same_labels)

Qs = jnp.asarray(np.stack([np.linalg.qr(wa)[0] for wa, _ in walkers]))
batched = jax.vmap(lambda q: m.channel_mps(q, plan, bond_plan)[0])(Qs)
print("one vmapped conversion of all six walkers, site tensors:", [tuple(A.shape) for A in batched])

distinct tensor shapes over the six walkers: 1
every walker's bond labels equal the dry run's: True
one vmapped conversion of all six walkers, site tensors: [(6, 1, 2, 2), (6, 2, 2, 4), (6, 4, 2, 4), (6, 4, 2, 4), (6, 4, 2, 4), (6, 4, 2, 4), (6, 4, 2, 2), (6, 2, 2, 1)]


For contrast, the cell runs the dry run on each walker itself: the counts that walker would choose if it were allowed to. They differ from the reference's at some splits, and every such difference changes the labels, and with them all later shapes. A per-walker allocation would therefore mean a separate compiled program for each walker, and a separate contraction layout with the trial.

In [14]:
print("counts each walker would choose in its own dry run, against the reference's:")
for k, (wa, _) in enumerate(walkers):
    own = m.plan_bonds(np.linalg.qr(wa)[0], plan, chi)
    diff = [i for i, (a, b) in enumerate(zip(own.kept_per_sector, bond_plan.kept_per_sector)) if a != b]
    first = (f"; first at split {diff[0]}: reference keeps {bond_plan.kept_per_sector[diff[0]]}, "
             f"walker would keep {own.kept_per_sector[diff[0]]}") if diff else ""
    print(f"walker {k}: differs at {len(diff):2d} of {len(own.kept_per_sector)} splits{first}")

counts each walker would choose in its own dry run, against the reference's:
walker 0: differs at  0 of 16 splits
walker 1: differs at  3 of 16 splits; first at split 11: reference keeps (0, 1, 2, 1), walker would keep (0, 2, 1, 1)
walker 2: differs at  3 of 16 splits; first at split 11: reference keeps (0, 1, 2, 1), walker would keep (1, 2, 1, 0)
walker 3: differs at  2 of 16 splits; first at split 11: reference keeps (0, 1, 2, 1), walker would keep (1, 2, 1, 0)
walker 4: differs at  0 of 16 splits
walker 5: differs at  1 of 16 splits; first at split 14: reference keeps (0, 1, 2, 1), walker would keep (0, 2, 1, 1)


### 5.4 ⚠ Approximation C: the price of frozen counts

Frozen counts make the conversion one static computation. The price is that a walker cannot rebalance its $\chi_w$ among the charge blocks when it has drifted from the reference. In the last cell, walkers 0 and 4 would have chosen exactly the reference's counts, so for them nothing is lost; walkers 1, 2, 3 and 5 would have chosen differently at a few splits.

The cell repeats the $\chi_w=4$ comparison of §5.1 with a third column: each walker converted with counts chosen from its own dry run. That is the best gate-by-gate truncation, but it is not usable in production, because the shapes would differ from walker to walker. It also checks that on the reference, the dry run's `reference_discarded_weight` is the norm actually lost, which is only true because of §5.1.

In [15]:
print(f"chi_w = {chi}: 1 - fidelity with the exact channel MPS")
print(f"{'walker':>6} {'counts frozen on reference':>27} {'own counts (not jit-able)':>26} {'best possible':>15}")
for k, (wa, _) in enumerate(walkers):
    Qk = jnp.asarray(np.linalg.qr(wa)[0])
    exact_k, _, _ = m.channel_mps(Qk, plan)
    frozen, frozen_charges, _ = m.channel_mps(Qk, plan, bond_plan)
    own, _, _ = m.channel_mps(Qk, plan, m.plan_bonds(np.asarray(Qk), plan, chi))
    best = canonical_compress(exact_k, [len(q) for q in frozen_charges])
    print(f"{k:>6} {1 - fidelity(exact_k, frozen):27.2e} {1 - fidelity(exact_k, own):26.2e} "
          f"{1 - fidelity(exact_k, best):15.2e}")

ref_exact, _, _ = m.channel_mps(jnp.asarray(C), plan)
ref_trunc, _, _ = m.channel_mps(jnp.asarray(C), plan, bond_plan)
print(f"reference: discarded weight from the dry run {bond_plan.reference_discarded_weight:.3e}, "
      f"actual 1 - F {1 - fidelity(ref_exact, ref_trunc):.3e}")

chi_w = 4: 1 - fidelity with the exact channel MPS
walker  counts frozen on reference  own counts (not jit-able)   best possible
     0                    5.19e-04                   5.19e-04        5.19e-04
     1                    1.07e-02                   1.28e-03        1.28e-03
     2                    1.97e-03                   5.47e-04        5.47e-04
     3                    2.65e-04                   9.86e-05        9.86e-05
     4                    3.81e-05                   3.81e-05        3.81e-05
     5                    2.19e-04                   1.41e-04        1.41e-04
reference: discarded weight from the dry run 5.909e-03, actual 1 - F 5.794e-03


With its own counts, every walker reaches exactly the best achievable value. So the whole gap left in §5.1 is Approximation C, and truncating gate by gate with the centre on each gate loses nothing against compressing the exact state once.

**Measured at production size.** From the review of `mps_cpmc_new.py`, not recomputed here: $L=16$, $\chi_T=64$, 24 CPMC walkers, exact enumeration over $12870^2$ determinant pairs. Median relative overlap error:

| $\chi_w$ (d=4 bond) | centre not moved (old) | centre on each gate (now) |
|---|---|---|
| 20 (400) | 5.8e-3 | 6.9e-8 |
| 12 (144) | 1.6e-2 | 5.6e-6 |
| 8 (64) | 9.1e-2 | 9.1e-5 |
| 6 (36) | 1.7e-1 | 5.3e-4 |
| 4 (16) | 4.9e-1 | 2.7e-2 |

Removing C would mean a per-walker allocation, which needs padded blocks under JIT. At the padded bond dimension it would need, the frozen allocation is 16 to 60 times more accurate. So C is not worth removing; raising $\chi_w$ is the better use of the same bond dimension.

## 6. Two spin channels, one MPS: the reordering sign

The $\uparrow$ and $\downarrow$ channels are converted separately, but the trial lives in the local basis $l = n_\uparrow + 2n_\downarrow$ with the site-by-site fermion order of §1. The product of the two channel states is in the order "all $\uparrow$, then all $\downarrow$". Reordering gives the sign $(-1)^{K(\mathbf n)}$ of §2, and $K$ splits over sites:

\begin{equation*}
(-1)^{K(\mathbf n)} = \prod_x (-1)^{\,n_{x\uparrow}\,N_{\downarrow}(<x)},
\end{equation*}

where $N_\downarrow(<x)$, the number of $\downarrow$ electrons left of site $x$, is exactly the charge label of the $\downarrow$ channel's bond to the left of $x$. So the sign is a local factor on each site tensor, and `combine_channels` builds

\begin{equation*}
W^{[x]}_{(a,a'),\,n_\uparrow+2n_\downarrow,\,(b,b')} = (-1)^{\,n_\uparrow\,q^\downarrow_{a'}}\,A^{[x]}_{a,n_\uparrow,b}\,A'^{[x]}_{a',n_\downarrow,b'},
\end{equation*}

with $A$ the $\uparrow$ channel, $A'$ the $\downarrow$ channel, and $q^\downarrow_{a'}$ the charge of bond index $a'$.

The production code never forms this $d=4$ MPS, whose bond is $\chi_w^2$. It extracts only the charge blocks the trial can see (`extract_channel_blocks`) and contracts them as padded dense blocks (`blocked_contract_from_blocks`). That is exact and returns the same number.

In [16]:
wa, wb = walkers[0]
Qa, Ra = np.linalg.qr(wa)
Qb, Rb = np.linalg.qr(wb)
alpha, qa, ga = m.channel_mps(jnp.asarray(Qa), plan)
beta, qb, gb = m.channel_mps(jnp.asarray(Qb), plan)
combined, combined_charges = m.combine_channels(alpha, qa, beta, qb)      # d = 4, with the sign
combined = [np.asarray(A) for A in combined]

i, j = 5, 17                                                              # any two strings
lhs = amplitude(combined, occ[i], occ[j])
rhs = reorder_sign(occ[i], occ[j]) * np.linalg.det(Qa[strings[i]]) * np.linalg.det(Qb[strings[j]]) / float(ga * gb)
print("one amplitude of W, sign included:", np.isclose(lhs, rhs))

prefactor = np.linalg.det(Ra) * np.linalg.det(Rb) * float(ga * gb)
H_trial_j = tuple(jnp.asarray(A) for A in H_trial)
ops = m.make_walker_ops(C, C, plan, plan, None, None, trial, trial_charges, H_trial_j)
print(f"det(R_a) det(R_b) g_a g_b <T|W>  {prefactor * mps_dot(trial, combined): .12e}")
print(f"production overlap (charge blocks) {float(ops.overlap((jnp.asarray(wa), jnp.asarray(wb)))): .12e}")
print(f"brute force                        {exact_overlap(wa, wb): .12e}")

one amplitude of W, sign included: True
det(R_a) det(R_b) g_a g_b <T|W>  -9.049040051784e+06
production overlap (charge blocks) -9.049040051784e+06
brute force                        -9.049040051784e+06


## 7. The overlap CPMC uses

Putting the pieces together,

\begin{equation*}
\bra{\psi_T}\phi\rangle = \underbrace{\det R_\uparrow\det R_\downarrow}_{\text{QR, §3.1}}\;\underbrace{g_\uparrow g_\downarrow}_{\text{gauge, §3.3}}\;\bra{\psi_T}W\rangle,
\end{equation*}

with $W$ the combined walker MPS. The prefactor comes from the walker's own determinants, never from a reference amplitude. `make_walker_ops` bundles the conversion, this overlap, the local energy (§8) and the fast sweep (§9), exactly as `main()` uses them.

With `walker_channel_chi=None` everything is exact. With truncation, $W$ becomes $\tilde W$ and the overlap is approximate (**B**, **C**).

## 8. The local energy

$H\ket{\psi_T}$ is built once as an MPS: the Hubbard MPO (bond dimension 6) applied to the trial and compressed with a relative singular-value cutoff of $10^{-13}$, which is effectively exact. The local energy of a walker is

\begin{equation*}
E_L(\phi) = \frac{\bra{\psi_T}H\ket{\phi}}{\bra{\psi_T}\phi\rangle} = \frac{\bra{H\psi_T}\tilde W\rangle}{\bra{\psi_T}\tilde W\rangle}.
\end{equation*}

The prefactor cancels. Numerator and denominator use the **same** walker MPS $\tilde W$, so with truncation $E_L$ is the local energy of $\tilde W$, not of $\phi$. The table shows both errors against brute force, for all six walkers.

In [17]:
def walker_ops(chi):
    bonds = None if chi is None else m.plan_bonds(C, plan, chi)
    return m.make_walker_ops(C, C, plan, plan, bonds, bonds, trial, trial_charges, H_trial_j)


print(f"{'chi_w':>6} {'d=4 bond':>9} {'median |dO|/|O|':>16} {'max |dO|/|O|':>13} {'max |dE_L|':>11}")
for chi in (None, 8, 6, 4, 3, 2):
    ops = walker_ops(chi)
    dO, dE = [], []
    for wa, wb in walkers:
        w = (jnp.asarray(wa), jnp.asarray(wb))
        dO.append(abs(float(ops.overlap(w)) - exact_overlap(wa, wb)) / abs(exact_overlap(wa, wb)))
        dE.append(abs(float(ops.energy(w)) - exact_energy(wa, wb)))
    print(f"{str(chi):>6} {max(map(len, ops.walker_charges)):>9} {np.median(dO):16.1e} {max(dO):13.1e} {max(dE):11.1e}")

 chi_w  d=4 bond  median |dO|/|O|  max |dO|/|O|  max |dE_L|
  None       256          1.9e-15       1.4e-14     2.7e-15
     8        64          3.7e-06       3.7e-05     2.7e-05
     6        36          1.3e-04       6.1e-04     1.5e-04
     4        16          5.2e-03       2.7e-02     9.7e-03
     3         9          3.5e-02       1.5e-01     4.6e-03
     2         4          3.0e-01       7.7e-01     5.4e-02


## 9. The CPMC step and the fast sweep

A step is $e^{-\Delta\tau K/2}\,e^{-\Delta\tau V}\,e^{-\Delta\tau K/2}$, as in `cpmc.ipynb`. The one-body halves are dense rotations of the walker, and each is followed by one overlap from a fresh conversion. The interaction is sampled site by site with the discrete Hubbard–Stratonovich field of `cpmc.ipynb`: on site $x$, field $s$ multiplies row $x$ of $\phi_\uparrow$ and of $\phi_\downarrow$ by the constants $h_{s\uparrow}$ and $h_{s\downarrow}$ (`hs_constant`, which includes the factor $e^{-\Delta\tau U/2}$). In the local basis this is a **diagonal** one-site operator,

\begin{equation*}
\hat B_x(s) = \mathrm{diag}\big(1,\ h_{s\uparrow},\ h_{s\downarrow},\ h_{s\uparrow}h_{s\downarrow}\big)\qquad\text{on } l = 0,1,2,3 .
\end{equation*}

It acts identically on $\phi$ and on its MPS, and it leaves the prefactor of §7 unchanged. So one conversion per step is enough. Store right environments $R^{[x+1]}$ (sites $x+1,\dots$ contracted with the trial), and carry a left environment $L^{[x]}$ that already contains the fields chosen on the sites $<x$. The *marginal*

\begin{equation*}
M_l = \sum L^{[x]}\,W^{[x]}_{l}\,T^{[x]}_{l}\,R^{[x+1]}\qquad (T = \text{trial})
\end{equation*}

gives the overlap after either field as $\sum_l B_x(s)_{ll}M_l$, so one contraction gives the ratio for both $s$. The chosen field is then folded into $L^{[x+1]}$. The rest is trot's `cpmc_step`: each field is proposed with probability $\propto\frac12\,\mathrm{ratio}$ after the constraint rule `constrain_ratio` (identical to trot's: a ratio at or below `weight_floor`, which includes every sign change, becomes zero), and the walker's weight is multiplied by the sum.

The cell is a readable version of that loop on the dense $d=4$ walker MPS of §6. With exact conversion it picks the same fields and ends at the same overlap as the production `sweep`, and that final overlap equals a fresh conversion of the updated walker.

In [18]:
gamma = np.arccosh(np.exp(0.5 * dt * U))
hs = np.exp(-0.5 * dt * U) * np.array([[np.exp(gamma), np.exp(-gamma)],   # field 0: row factors (up, down)
                                       [np.exp(-gamma), np.exp(gamma)]])  # field 1
D = np.array([[1.0, h[0], h[1], h[0] * h[1]] for h in hs])                 # D[field, l]
floor = 1e-8


def readable_sweep(W, prefactor, randoms):
    right = [np.ones((1, 1))]                                   # right[x]: sites x..L-1 with the trial
    for Wx, Tx in zip(reversed(W), reversed(trial)):
        right.insert(0, np.einsum("asc,bsd,cd->ab", Wx, Tx, right[0]))
    left, overlap, weight, fields = np.ones((1, 1)), prefactor * right[0][0, 0], 1.0, []
    for x, (Wx, Tx) in enumerate(zip(W, trial)):
        marginal = np.einsum("ab,alc,bld,cd->l", left, Wx, Tx, right[x + 1])
        ratios = prefactor * (D @ marginal) / overlap           # both fields at once
        ratios = np.where(ratios <= floor, 0.0, ratios)         # the constraint, as in trot
        probs = 0.5 * ratios
        field = 0 if randoms[x] < probs[0] / probs.sum() else 1
        weight *= probs.sum()
        overlap = prefactor * D[field] @ marginal
        left = np.einsum("ab,alc,bld,l->cd", left, Wx, Tx, D[field])   # fold site x in, with its field
        fields.append(field)
    return fields, overlap, weight


randoms = rng.random(L)
fields, overlap_end, weight = readable_sweep(combined, prefactor, randoms)

ops = walker_ops(None)
wa, wb = walkers[0]
ca2, cb2, before, after, wfac, _ = ops.sweep(jnp.asarray(wa), jnp.asarray(wb), jnp.asarray(randoms), jnp.asarray(hs), floor)
fields_module = [0 if np.isclose(float(ca2[x, 0] / wa[x, 0]), hs[0, 0]) else 1 for x in range(L)]
print("fields, readable  :", fields)
print("fields, production:", fields_module)
print(f"overlap after the loop: readable {overlap_end:.12e}, production {float(after):.12e}, "
      f"fresh conversion {float(ops.overlap((ca2, cb2))):.12e}")
print(f"weight factor: readable {weight:.12f}, production {float(wfac):.12f}")

fields, readable  : [1, 0, 1, 1, 1, 0, 0, 1]
fields, production: [1, 0, 1, 1, 1, 0, 0, 1]
overlap after the loop: readable -9.653312906889e+06, production -9.653312906889e+06, fresh conversion -9.653312906889e+06
weight factor: readable 0.958868617068, production 0.958868617069


### ⚠ Approximation D: the site loop works on the start-of-step walker MPS

With truncation, the sweep evaluates every field ratio on $\hat B\,\tilde W$, where $\tilde W$ is the truncated MPS of the walker *before* the loop. The next overlap, after the second one-body half, comes from a fresh conversion of the updated walker, i.e. from the truncated MPS of $\hat B\phi$. These are two different approximations of the same state, so each step the weights pick up their ratio instead of telescoping exactly. With exact conversion both are exact and agree to $10^{-15}$ (above). The cell measures the mismatch. It shrinks with $\chi_w$ as fast as the truncation error itself.

In [19]:
print(f"{'chi_w':>6} {'median |sweep end - fresh conversion| / |O|':>45}")
for chi in (None, 8, 4, 2):
    ops = walker_ops(chi)
    mismatch = []
    for wa, wb in walkers:
        ca2, cb2, _, after, _, _ = ops.sweep(jnp.asarray(wa), jnp.asarray(wb), jnp.asarray(randoms), jnp.asarray(hs), floor)
        fresh = float(ops.overlap((ca2, cb2)))
        mismatch.append(abs(float(after) - fresh) / abs(fresh))
    print(f"{str(chi):>6} {np.median(mismatch):45.1e}")

 chi_w   median |sweep end - fresh conversion| / |O|
  None                                       1.3e-15
     8                                       2.0e-06
     4                                       2.7e-04
     2                                       1.7e-02


## 10. Summary: where the algorithm is approximate

| | approximation | where in `mps_cpmc_new.py` | measured size | control |
|---|---|---|---|---|
| **A** | Fishman–White blocks not rank-exact for walkers (`adaptive` only) | `make_orbital_plan`, `channel_angles` | dropped norm $\lesssim 10^{-9}$ on CPMC walkers up to $L=48$; zero with `rank_exact` | `orbital_plan` |
| **B** | walker MPS truncated to $\chi_w$ per channel | `split_pair` with `kept`, `plan_bonds` | the dominant error; §5 table ($L=16$: $7\times10^{-8}$ at $\chi_w=20$, $3\times10^{-2}$ at $\chi_w=4$) | `walker_channel_chi` |
| **C** | kept counts per charge block frozen on the reference | `plan_bonds` → `BondPlan.kept_per_sector` | up to a few times B's error on strongly drifted walkers (§5.4); not worth removing | the reference determinant |
| **D** | site-loop ratios use the start-of-step walker MPS | `make_fast_sweep` | $8\times10^{-6}$ relative at $\chi_w=8$ here ($L=8$), $4\times10^{-6}$ at $L=12$ | $\chi_w$ |
| — | the trial is a finite-$\chi_T$ DMRG state | `run_dmrg` | sets the constraint bias, not an overlap error | `trial_chi` |
| — | Trotter step and constraint | trot, `make_fast_prop_ops` | standard CPMC | `dt` |

**Exact to machine precision, and checked above:** the local basis and fermion conventions, the Givens gates and their many-body form, the gauge $\det R\,g$, the reordering sign, the charge-blocked contraction, $H\ket{\psi_T}$, the environment sweep, and the constraint rule. The rule is identical to trot's `cpmc_step`; `tests/test_gmps_mps_cpmc.py` checks the fast step against it step for step.

**How the approximations enter CPMC.** The importance function, the constraint and the local energy are all evaluated on $\tilde\phi$ instead of $\phi$. The estimator is then not the mixed estimate of any fixed trial, and it is not variational: it can land below the exact energy, as the $\chi_w=2$ runs did. B, C and D all vanish as $\chi_w$ grows, and A is absent with `rank_exact`.